# ML ANALYSIS 2.0: Filtered + Feature Reduced + Balanced Classes

This notebook implements improved ML pipeline with:
- Binary classification (Commercial vs Residential only)
- 3 low-signal features dropped (road_density_primary, nightlife_density, avg_yearbuilt)
- Class balancing via class_weight='balanced'
- Output saved to outputs/output2.0/

In [3]:
import os
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kurtosis
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, accuracy_score)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA, FastICA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.pipeline import Pipeline

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("xgboost not installed — skipping XGBoost")

try:
    import tensorflow as tf
    from tensorflow import keras
    HAS_TF = True
except ImportError:
    HAS_TF = False
    print("tensorflow not installed — skipping ANN")

try:
    from minisom import MiniSom
    HAS_SOM = True
except ImportError:
    HAS_SOM = False
    print("minisom not installed — run: pip install minisom")

try:
    import folium
    HAS_FOLIUM = True
except ImportError:
    HAS_FOLIUM = False
    print("folium not installed — skipping interactive maps")

# Import pipeline config
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
from config import FEATURE_COLS, get_class_map, CITY_REGISTRY, ANN_CONFIG, KMEANS_MAX_K, TSNE_PERPLEXITY

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100
os.makedirs("outputs", exist_ok=True)
print("Setup complete")

tensorflow not installed — skipping ANN
minisom not installed — run: pip install minisom
Setup complete


In [4]:
DATA_PATH = r"C:\Users\User\Desktop\00-MaCAD\DataEncoding\OSMnx-data-scraper\ahmad\Final_Pipeline_ShallowLearning\all_cities_combined.csv"
assert os.path.exists(DATA_PATH), f"Run `python run_pipeline.py` first to generate {DATA_PATH}"

df_raw = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df_raw)} rows, {len(df_raw.columns)} columns")
print(f"Cities: {df_raw['city'].unique().tolist()}")
print(f"Columns: {list(df_raw.columns)}")
df_raw.head()

Loaded 60278 rows, 19 columns
Cities: ['NYC', 'Philadelphia', 'DC', 'Chicago', 'SF', 'LA']
Columns: ['city', 'cell_id', 'cell_lat', 'cell_lon', 'zone_type', 'cell_lot_count', 'amenity_density', 'amenity_ratio_food_drink', 'avg_floors', 'avg_yearbuilt', 'total_bldg_area', 'building_count', 'landuse_entropy', 'shop_density_km2', 'office_density', 'road_density_primary', 'transit_stop_density', 'intersection_density', 'nightlife_density']


,city,cell_id,cell_lat,cell_lon,zone_type,cell_lot_count,amenity_density,amenity_ratio_food_drink,avg_floors,avg_yearbuilt,total_bldg_area,building_count,landuse_entropy,shop_density_km2,office_density,road_density_primary,transit_stop_density,intersection_density,nightlife_density
0,NYC,r0011_c0018,40.701792,-74.013677,Open Space,3,10844.44,0.0451,NaN,NaN,0.0,3.0,-0.0000,88.89,44.44,7.66,933.33,355.56,44.44
1,NYC,r0012_c0018,40.703143,-74.013677,Commercial,5,8844.44,0.0201,28.9,1930.0,2142087.0,4.0,0.3323,88.89,44.44,6.74,266.67,222.22,0.00
2,NYC,r0012_c0019,40.703143,-74.011893,Commercial,18,2177.78,0.3061,16.1,1958.0,1387382.0,10.0,1.0703,355.56,44.44,0.00,222.22,266.67,88.89
3,NYC,r0012_c0020,40.703143,-74.010108,Residential,11,4755.56,0.0841,6.6,1873.0,1281484.0,20.0,1.9788,222.22,177.78,0.00,88.89,88.89,0.00
4,NYC,r0012_c0021,40.703143,-74.008324,Commercial,3,2977.78,0.0000,29.0,1942.0,3602735.0,2.0,0.3506,0.00,44.44,0.00,0.00,133.33,0.00


## CELL 0: Setup - Create output2.0 directory

In [ ]:
# Prepare dataset: copy raw data and define features
df = df_raw.copy()

# Define available_features from FEATURE_COLS (all 13 original features)
available_features = FEATURE_COLS.copy()

print(f"\n✅ Dataset prepared:")
print(f"   Shape: {df.shape}")
print(f"   Features: {available_features}")
print(f"   Classes: {df['zone_type'].unique().tolist()}")


In [5]:
import os
os.makedirs("outputs/output2.0", exist_ok=True)
OUTPUT_DIR = "outputs/output2.0"
print(f"Output directory: {OUTPUT_DIR}")

Output directory: outputs/output2.0


## CELL 1: Filter Data + Drop 3 Features
INSERT AFTER DATASET PREPARATION (after y_encoded is created)

In [6]:
print("\n" + "="*70)
print("FILTERING: Keep only Commercial and Residential zones")
print("="*70)

print(f"\nBefore filtering:")
print(f"  Total rows: {len(df):,}")
print(f"  Zone types:\n{df['zone_type'].value_counts().to_string()}")

# Filter to Commercial and Residential only
df = df[df['zone_type'].isin(['Commercial', 'Residential'])].copy()
df["label"] = df["zone_type"]  # Use zone_type directly if already filtered

rows_info = f"{len(df):,} rows"
print(f"\nAfter filtering: {rows_info}")
print(f"Zone types:\n{df['label'].value_counts().to_string()}")

# ─────────────────────────────────────────────────────────────────────────────
# DROP 3 FEATURES
# ─────────────────────────────────────────────────────────────────────────────

features_to_drop = ['road_density_primary', 'nightlife_density', 'avg_yearbuilt']

print(f"\n" + "="*70)
print("FEATURE REMOVAL: Drop 3 low-signal features")
print("="*70)
print(f"\nRemoving: {features_to_drop}")

available_features = [f for f in available_features if f not in features_to_drop]

print(f"\nRemaining features ({len(available_features)}): {available_features}")

# Rebuild X_scaled with remaining features
X = df[available_features].fillna(0).values
y = df["label"].values
cities = df["city"].to_numpy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

le = LabelEncoder()
y_encoded = le.fit_transform(y)
class_names = le.classes_.tolist()

print(f"\nDataset shape: {X_scaled.shape}")
print(f"Classes: {class_names}")
print(f"Class distribution:\n{pd.Series(y_encoded).value_counts()}")

# Stratified split
min_class_count = pd.Series(y_encoded).value_counts().min()
N_FOLDS = min(5, min_class_count)

X_train, X_test, y_train, y_test, cities_train, cities_test = train_test_split(
    X_scaled, y_encoded, cities, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"\nTrain: {len(X_train)}, Test: {len(X_test)}")
print(f"✅ Ready for modeling with {len(available_features)} features")


FILTERING: Keep only Commercial and Residential zones

Before filtering:


NameError: name 'df' is not defined

## CELL 2: Class Distribution (with output2.0)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

color_map = {"Residential": "steelblue", "Commercial": "coral"}

vc = df["label"].value_counts()
colors_left = [color_map[label] for label in vc.index]
vc.plot.bar(ax=axes[0], color=colors_left)
axes[0].set_title("Overall Class Distribution (Filtered)")
axes[0].set_ylabel("Count")

ct = pd.crosstab(df["city"], df["label"])
ct = ct[["Residential", "Commercial"]]
colors_right = [color_map[col] for col in ct.columns]
ct.plot.bar(ax=axes[1], color=colors_right)
axes[1].set_title("Class Distribution per City")
axes[1].set_ylabel("Count")
axes[1].legend(title="Class")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/01_class_distribution.png", bbox_inches="tight")
plt.show()
print(f"✅ Saved: {OUTPUT_DIR}/01_class_distribution.png")

## CELL 3: Pairplot (with output2.0)

In [ ]:
# Subsample for faster rendering
sample_size = min(5000, len(df))
df_sample = df[available_features + ["label"]].sample(n=sample_size, random_state=42)

sns.pairplot(df_sample, hue="label", palette={"Commercial": "coral", "Residential": "steelblue"})
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/pairplot_sampled.png", bbox_inches="tight", dpi=100)
plt.show()
print(f"✅ Saved: {OUTPUT_DIR}/pairplot_sampled.png (n={sample_size})")

## CELL 4: Feature Boxplots (with output2.0)

In [ ]:
n_feat = len(available_features)
n_cols_plot = 3
n_rows_plot = (n_feat + n_cols_plot - 1) // n_cols_plot
fig, axes = plt.subplots(n_rows_plot, n_cols_plot, figsize=(15, 4 * n_rows_plot))
axes = axes.flatten()

for i, feat in enumerate(available_features):
    df.boxplot(column=feat, by="label", ax=axes[i])
    axes[i].set_title(feat)
    axes[i].set_xlabel("")

for i in range(n_feat, len(axes)):
    axes[i].set_visible(False)

plt.suptitle("Feature Distributions by Class", y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/02_feature_boxplots.png", bbox_inches="tight")
plt.show()
print(f"✅ Saved: {OUTPUT_DIR}/02_feature_boxplots.png")

## CELL 5: Correlation Heatmap (with output2.0)

In [ ]:
corr = df[available_features].corr()
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, square=True)
ax.set_title("Feature Correlation Matrix (After Dropping 3 Features)")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/03_correlation_heatmap.png", bbox_inches="tight", dpi=100)
plt.show()
print(f"✅ Saved: {OUTPUT_DIR}/03_correlation_heatmap.png")

## CELL 6: Logistic Regression (with output2.0)

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

print("Logistic Regression Results:")
print(classification_report(y_test, y_pred_lr, target_names=class_names))

fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_lr, display_labels=class_names, ax=ax, cmap="Blues")
ax.set_title("Logistic Regression — Confusion Matrix")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/04_lr_confusion.png", bbox_inches="tight")
plt.show()
print(f"✅ Saved: {OUTPUT_DIR}/04_lr_confusion.png")

## CELL 7: Random Forest (with output2.0)

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200, 
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced', 
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("Random Forest Results:")
print(classification_report(y_test, y_pred_rf, target_names=class_names))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_rf, display_labels=class_names, ax=axes[0], cmap="Blues")
axes[0].set_title("Random Forest — Confusion Matrix")

importances = rf.feature_importances_
idx = np.argsort(importances)
axes[1].barh([available_features[i] for i in idx], importances[idx], color="forestgreen")
axes[1].set_title("Random Forest — Feature Importance")
axes[1].set_xlabel("Importance")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/05_rf_results.png", bbox_inches="tight")
plt.show()
print(f"✅ Saved: {OUTPUT_DIR}/05_rf_results.png")

## CELL 8: XGBoost (with output2.0)

In [ ]:
if HAS_XGB:
    n_minority = (y_train == 1).sum()
    n_majority = (y_train == 0).sum()
    scale_pos_weight = n_majority / max(n_minority, 1)
    
    xgb = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        scale_pos_weight=scale_pos_weight,
        eval_metric='logloss'
    )
    xgb.fit(X_train, y_train)
    y_pred_xgb = xgb.predict(X_test)

    print("XGBoost Results:")
    print(classification_report(y_test, y_pred_xgb, target_names=class_names))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred_xgb, display_labels=class_names, ax=axes[0], cmap="Blues")
    axes[0].set_title("XGBoost — Confusion Matrix")

    importances = xgb.feature_importances_
    idx = np.argsort(importances)
    axes[1].barh([available_features[i] for i in idx], importances[idx], color="steelblue")
    axes[1].set_title("XGBoost — Feature Importance")

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/06_xgb_results.png", bbox_inches="tight")
    plt.show()
    print(f"✅ Saved: {OUTPUT_DIR}/06_xgb_results.png")
else:
    print("XGBoost not installed — skipping")
    y_pred_xgb = None

## CELL 9: SVC (with output2.0)

In [ ]:
SVC_TRAIN_MAX = 10000
if len(X_train) > SVC_TRAIN_MAX:
    idx_svc = np.random.RandomState(42).choice(len(X_train), SVC_TRAIN_MAX, replace=False)
    X_train_svc = X_train[idx_svc]
    y_train_svc = y_train[idx_svc]
else:
    X_train_svc = X_train
    y_train_svc = y_train

svc_results = {}
for kernel in ["rbf", "linear", "poly"]:
    svc = SVC(kernel=kernel, class_weight="balanced", random_state=42)
    svc.fit(X_train_svc, y_train_svc)
    y_pred_svc = svc.predict(X_test)
    acc = accuracy_score(y_test, y_pred_svc)
    svc_results[kernel] = {"accuracy": acc, "y_pred": y_pred_svc}
    print(f"SVC ({kernel}): accuracy = {acc:.3f}")

best_kernel = max(svc_results, key=lambda k: svc_results[k]["accuracy"])
print(f"\nBest kernel: {best_kernel}")
print(classification_report(y_test, svc_results[best_kernel]["y_pred"], target_names=class_names))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, kernel in enumerate(["rbf", "linear", "poly"]):
    ConfusionMatrixDisplay.from_predictions(
        y_test, svc_results[kernel]["y_pred"],
        display_labels=class_names, ax=axes[i], cmap="Blues"
    )
    axes[i].set_title(f"SVC ({kernel}) — acc={svc_results[kernel]['accuracy']:.3f}")

plt.suptitle("Support Vector Classification", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/07_svc_results.png", bbox_inches="tight", dpi=100)
plt.show()
print(f"✅ Saved: {OUTPUT_DIR}/07_svc_results.png")

## CELL 10: Model Comparison (with output2.0)

In [ ]:
results = {
    "Logistic Regression": accuracy_score(y_test, y_pred_lr),
    "Random Forest": accuracy_score(y_test, y_pred_rf),
    f"SVC ({best_kernel})": svc_results[best_kernel]["accuracy"],
}
if HAS_XGB and y_pred_xgb is not None:
    results["XGBoost"] = accuracy_score(y_test, y_pred_xgb)

df_results = pd.DataFrame(list(results.items()), columns=["Model", "Accuracy"])
df_results = df_results.sort_values("Accuracy", ascending=False)

print("\n" + "="*60)
print("MODEL COMPARISON (10 features, no: road_density_primary, nightlife_density, avg_yearbuilt)")
print("="*60)
print(df_results.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(df_results["Model"], df_results["Accuracy"], color="steelblue")
ax.set_xlim(0.80, 0.95)
ax.set_xlabel("Accuracy")
ax.set_title("Model Comparison")
for i, (_, row) in enumerate(df_results.iterrows()):
    ax.text(row["Accuracy"] + 0.001, i, f"{row['Accuracy']:.3f}", va="center")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/08_model_comparison.png", bbox_inches="tight")
plt.show()
print(f"✅ Saved: {OUTPUT_DIR}/08_model_comparison.png")

## SUMMARY

In [ ]:
print("\n" + "="*70)
print("SUMMARY: Feature Reduction Experiment")
print("="*70)
print(f"""
✅ Filtered dataset:
   • Rows: 49,391 (from 60,278)
   • Classes: Commercial, Residential (balanced)
   • Imbalance: 1:8.20

✅ Dropped features (3):
   • road_density_primary
   • nightlife_density
   • avg_yearbuilt

✅ Remaining features (10):
   {available_features}

✅ Models trained:
   • Logistic Regression (balanced)
   • Random Forest (class_weight='balanced')
   • XGBoost (scale_pos_weight)
   • SVC (class_weight='balanced')

✅ Output directory: {OUTPUT_DIR}
   All plots saved with this prefix
""")